# AutoGluon Tabular Forecasting

## Import Library

In [116]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# For time series
from typing import List

# from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import userdata

In [117]:
def download_kaggle(kaggle_command="kaggle competitions download -c liver-fibrosis-severity-prediction"):

  # Get Kaggle Key
  kaggle_username = userdata.get("KAGGLE_USER")
  kaggle_key = userdata.get("KAGGLE_KEY")
  if not kaggle_username or not kaggle_key:
      print("Error: Kaggle_USERNAME or Kaggle_KEY not found in Colab Secrets.")
      return

  # Write the credentials to ~/.kaggle/kaggle.json
  kaggle_dir = os.path.expanduser("~/.kaggle")
  os.makedirs(kaggle_dir, exist_ok=True)

  # Create JSON
  kaggle_json_path = os.path.join(kaggle_dir, "kaggle.json")
  with open(kaggle_json_path, "w") as f:
      f.write(f'{{"username":"{kaggle_username}","key":"{kaggle_key}"}}')
  os.chmod(kaggle_json_path, 0o600)

  try:
      os.system(kaggle_command)
      print("\n--- Download complete! ---")
      os.system("ls -la")
      os.system("unzip -o '*.zip' && rm -f *.zip")
      os.system("ls -la")

  except Exception as e:
      print(f"An error occurred during download: {e}")

In [118]:
download_kaggle(
    "kaggle competitions download -c individual-test-temperature-prediction"
)


--- Download complete! ---


## Explore Dataset

In [119]:
df_train = pd.read_csv("/content/IOT_Train.csv")
df_train

,mac,station_name,tambon_code,tambon_namt,amphur_code,amphur_namt,province_code,province_namt,latitude,longitude,time,humid,light,pm10,pm2.5,rainfall,wind_direct,wind_speed,temp
0,3C71BF18EA64,บ้านนา_2,300903,กำปัง,3009,โนนไทย,30,นครราชสีมา,15.112831,102.052114,2022-07-20 15:00:00+07:00,70.6,59.0,NaN,NaN,0.0,45.0,4.3,32.6
1,3C71BF18EA64,บ้านนา_2,300903,กำปัง,3009,โนนไทย,30,นครราชสีมา,15.112831,102.052114,2022-05-16 09:00:00+07:00,62.3,76.0,NaN,NaN,0.0,45.0,1.9,37.9
2,3C71BF18EA64,บ้านนา_2,300903,กำปัง,3009,โนนไทย,30,นครราชสีมา,15.112831,102.052114,2022-06-26 19:00:00+07:00,90.3,0.0,NaN,NaN,0.0,135.0,3.2,26.4
3,3C71BF18EA64,บ้านนา_2,300903,กำปัง,3009,โนนไทย,30,นครราชสีมา,15.112831,102.052114,2022-07-16 04:00:00+07:00,91.7,0.0,NaN,NaN,0.0,157.5,0.0,25.6
4,3C71BF18EA64,บ้านนา_2,300903,กำปัง,3009,โนนไทย,30,นครราชสีมา,15.112831,102.052114,2022-05-30 08:00:00+07:00,59.1,71.0,NaN,NaN,0.0,180.0,2.3,36.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13630,3C71BF18CEA4,โรงเรียนหนองสูงสามัคคีวิทยา_2,490706,หนองสูงเหนือ,4907,หนองสูง,49,มุกดาหาร,16.494229,104.350891,2022-06-14 01:00:00+07:00,97.9,0.0,NaN,NaN,4.2,0.0,0.0,23.5
13631,3C71BF18CEA4,โรงเรียนหนองสูงสามัคคีวิทยา_2,490706,หนองสูงเหนือ,4907,หนองสูง,49,มุกดาหาร,16.494229,104.350891,2022-05-03 02:00:00+07:00,80.6,0.0,NaN,NaN,0.0,135.0,2.9,19.3
13632,3C71BF18CEA4,โรงเรียนหนองสูงสามัคคีวิทยา_2,490706,หนองสูงเหนือ,4907,หนองสูง,49,มุกดาหาร,16.494229,104.350891,2022-05-19 06:00:00+07:00,87.9,37.0,NaN,NaN,0.0,45.0,3.0,24.5
13633,3C71BF18CEA4,โรงเรียนหนองสูงสามัคคีวิทยา_2,490706,หนองสูงเหนือ,4907,หนองสูง,49,มุกดาหาร,16.494229,104.350891,2022-06-15 02:00:00+07:00,89.9,0.0,NaN,NaN,0.0,315.0,0.0,27.3


In [120]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13635 entries, 0 to 13634
Data columns (total 19 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   mac            13635 non-null  object 
 1   station_name   13635 non-null  object 
 2   tambon_code    13635 non-null  int64  
 3   tambon_namt    13635 non-null  object 
 4   amphur_code    13635 non-null  int64  
 5   amphur_namt    13635 non-null  object 
 6   province_code  13635 non-null  int64  
 7   province_namt  13635 non-null  object 
 8   latitude       13635 non-null  float64
 9   longitude      13635 non-null  float64
 10  time           13635 non-null  object 
 11  humid          13635 non-null  float64
 12  light          13635 non-null  float64
 13  pm10           6841 non-null   float64
 14  pm2.5          6841 non-null   float64
 15  rainfall       13635 non-null  float64
 16  wind_direct    13635 non-null  float64
 17  wind_speed     13635 non-null  float64
 18  temp  

In [121]:
print(len(df_train["mac"].unique()))
print(len(df_train["station_name"].unique()))

8
8


In [122]:
df_train["time"] = pd.to_datetime(df_train["time"])
df_train = df_train.sort_values(by="time")
df_train["time"]

,time
6361,2022-05-02 00:00:00+07:00
8429,2022-05-02 00:00:00+07:00
1678,2022-05-02 00:00:00+07:00
11871,2022-05-02 00:00:00+07:00
13614,2022-05-02 00:00:00+07:00
...,...
55,2022-07-31 21:00:00+07:00
10275,2022-07-31 21:00:00+07:00
1806,2022-07-31 21:00:00+07:00
8545,2022-07-31 21:00:00+07:00


In [123]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13635 entries, 6361 to 6830
Data columns (total 19 columns):
 #   Column         Non-Null Count  Dtype                    
---  ------         --------------  -----                    
 0   mac            13635 non-null  object                   
 1   station_name   13635 non-null  object                   
 2   tambon_code    13635 non-null  int64                    
 3   tambon_namt    13635 non-null  object                   
 4   amphur_code    13635 non-null  int64                    
 5   amphur_namt    13635 non-null  object                   
 6   province_code  13635 non-null  int64                    
 7   province_namt  13635 non-null  object                   
 8   latitude       13635 non-null  float64                  
 9   longitude      13635 non-null  float64                  
 10  time           13635 non-null  datetime64[ns, UTC+07:00]
 11  humid          13635 non-null  float64                  
 12  light          13635 

### Time Features

In [124]:
def extract_time_features(df: pd.DataFrame, time_columns: list = None) -> pd.DataFrame:
    """
    Extracts time-based features from specified columns in a DataFrame.
    If time_columns is not provided, it attempts to infer time columns
    by checking common time-related column names.

    Args:
        df (pd.DataFrame): The input DataFrame.
        time_columns (list, optional): A list of column names containing time data.
                                        If None, common time-related column names will be checked.

    Returns:
        pd.DataFrame: The DataFrame with new time feature columns.
    """

    df_copy = df.copy()

    # Common time-related column name patterns for inference
    if time_columns is None:
        possible_time_col_patterns = [
            'time', 'date', 'timestamp', 'datetime', 'created_at',
            'updated_at', 'ts', 'event_time', 'start_time', 'end_time'
        ]
        inferred_time_columns = [
            col for col in df_copy.columns
            if any(pattern in col.lower() for pattern in possible_time_col_patterns)
        ]
        if not inferred_time_columns:
            print("Warning: No time columns specified and no common time-related columns found. "
                  "Returning original DataFrame.")
            return df_copy
        time_columns = inferred_time_columns
        print(f"Inferred time columns: {time_columns}")

    # Possible time formats to try for robust parsing
    # Ordered from most specific/common to more general
    possible_time_formats = [
        "%Y-%m-%d %H:%M:%S.%f",  # 2025-07-05 13:42:29.123456
        "%Y-%m-%d %H:%M:%S",    # 2025-07-05 13:42:29
        "%Y-%m-%dT%H:%M:%S.%fZ",# ISO 8601 with Z (UTC)
        "%Y-%m-%dT%H:%M:%S",    # ISO 8601 without Z
        "%Y/%m/%d %H:%M:%S",    # 2025/07/05 13:42:29
        "%d-%m-%Y %H:%M:%S",    # 05-07-2025 13:42:29
        "%m/%d/%Y %I:%M:%S %p", # 07/05/2025 01:42:29 PM
        "%d/%m/%Y %H:%M",       # 05/07/2025 13:42
        "%Y-%m-%d",             # 2025-07-05
        "%d-%m-%Y",             # 05-07-2025
        "%m/%d/%Y",             # 07/05/2025
        "%H:%M:%S",             # 13:42:29
        "%H:%M",                # 13:42
        "%I:%M %p"              # 01:42 PM
    ]

    for col in time_columns:
        if col not in df_copy.columns:
            print(f"Warning: Column '{col}' not found in DataFrame. Skipping.")
            continue

        # Convert to datetime, coercing errors to NaT (Not a Time)
        # We try multiple formats with errors='coerce' to handle mixed formats
        df_copy[f'{col}_dt'] = pd.to_datetime(df_copy[col], errors='coerce', infer_datetime_format=True)

        # Drop rows where conversion failed for this column, or handle as NaT
        # For simplicity in feature extraction, we'll proceed with NaT values
        # If a column is entirely NaT after conversion, it's probably not a time column
        if df_copy[f'{col}_dt'].isnull().all() and not df_copy[col].isnull().all():
            print(f"Warning: Column '{col}' could not be converted to datetime. Skipping feature extraction for this column.")
            df_copy = df_copy.drop(columns=[f'{col}_dt'])
            continue

        # Extract features
        dt_col = df_copy[f'{col}_dt'].dt
        df_copy[f'{col}_year'] = dt_col.year
        df_copy[f'{col}_month'] = dt_col.month
        df_copy[f'{col}_day'] = dt_col.day
        df_copy[f'{col}_hour'] = dt_col.hour
        df_copy[f'{col}_minute'] = dt_col.minute
        df_copy[f'{col}_second'] = dt_col.second
        df_copy[f'{col}_dayofweek'] = dt_col.dayofweek # Monday=0, Sunday=6
        df_copy[f'{col}_dayofyear'] = dt_col.dayofyear
        df_copy[f'{col}_weekofyear'] = dt_col.isocalendar().week.astype(int)
        df_copy[f'{col}_quarter'] = dt_col.quarter
        df_copy[f'{col}_is_weekend'] = dt_col.dayofweek.isin([5, 6]).astype(int)
        df_copy[f'{col}_season'] = (dt_col.month % 12 + 3) // 3 # Simple season mapping
        df_copy[f'{col}_is_month_start'] = dt_col.is_month_start.astype(int)
        df_copy[f'{col}_is_month_end'] = dt_col.is_month_end.astype(int)
        df_copy[f'{col}_is_quarter_start'] = dt_col.is_quarter_start.astype(int)
        df_copy[f'{col}_is_quarter_end'] = dt_col.is_quarter_end.astype(int)
        df_copy[f'{col}_is_year_start'] = dt_col.is_year_start.astype(int)
        df_copy[f'{col}_is_year_end'] = dt_col.is_year_end.astype(int)

        # Optional: Add cyclical features (sin/cos transformations for hour, dayofweek, etc.)
        # These are useful for models to understand cyclical patterns without arbitrary breaks
        # Example for hour
        df_copy[f'{col}_hour_sin'] = np.sin(2 * np.pi * dt_col.hour / 24)
        df_copy[f'{col}_hour_cos'] = np.cos(2 * np.pi * dt_col.hour / 24)

    return df_copy

In [125]:
df_train = extract_time_features(df=df_train, time_columns=["time"])
df_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13635 entries, 6361 to 6830
Data columns (total 40 columns):
 #   Column                 Non-Null Count  Dtype                    
---  ------                 --------------  -----                    
 0   mac                    13635 non-null  object                   
 1   station_name           13635 non-null  object                   
 2   tambon_code            13635 non-null  int64                    
 3   tambon_namt            13635 non-null  object                   
 4   amphur_code            13635 non-null  int64                    
 5   amphur_namt            13635 non-null  object                   
 6   province_code          13635 non-null  int64                    
 7   province_namt          13635 non-null  object                   
 8   latitude               13635 non-null  float64                  
 9   longitude              13635 non-null  float64                  
 10  time                   13635 non-null  datetime64

/tmp/ipython-input-124-1301132536.py:61: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df_copy[f'{col}_dt'] = pd.to_datetime(df_copy[col], errors='coerce', infer_datetime_format=True)


In [126]:
df_train = df_train.drop(columns=["time_dt", "mac"])
df_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13635 entries, 6361 to 6830
Data columns (total 38 columns):
 #   Column                 Non-Null Count  Dtype                    
---  ------                 --------------  -----                    
 0   station_name           13635 non-null  object                   
 1   tambon_code            13635 non-null  int64                    
 2   tambon_namt            13635 non-null  object                   
 3   amphur_code            13635 non-null  int64                    
 4   amphur_namt            13635 non-null  object                   
 5   province_code          13635 non-null  int64                    
 6   province_namt          13635 non-null  object                   
 7   latitude               13635 non-null  float64                  
 8   longitude              13635 non-null  float64                  
 9   time                   13635 non-null  datetime64[ns, UTC+07:00]
 10  humid                  13635 non-null  float64   

In [127]:
# df_train = df_train.drop(columns=["time"])
# df_train.info()

In [128]:
print(df_train["station_name"].unique())
print(df_train["tambon_namt"].unique())
print(df_train["province_namt"].unique())

['โรงเรียนท่าข้ามวิทยา' 'โรงเรียนบ้านนา' 'บ้านนา_2'
 'โรงเรียนสามเสนนอก(ประชาราษฎร์อนุกูล)' 'โรงเรียนหนองสูงสามัคคีวิทยา_2'
 'บ้านนาสะแบง_2' 'บ้านสำโรงเกียรติ_2' 'โรงเรียนสรรพวิทยาคม']
['ท่าข้าม' 'สถาน' 'กำปัง' 'ดินแดง' 'หนองสูงเหนือ' 'นาสะแบง' 'บักดอง'
 'แม่สอด']
['ชุมพร' 'น่าน' 'นครราชสีมา' 'กรุงเทพมหานคร' 'มุกดาหาร' 'บึงกาฬ'
 'ศรีสะเกษ' 'ตาก']


In [129]:
df_train

,station_name,tambon_code,tambon_namt,amphur_code,amphur_namt,province_code,province_namt,latitude,longitude,time,...,time_is_weekend,time_season,time_is_month_start,time_is_month_end,time_is_quarter_start,time_is_quarter_end,time_is_year_start,time_is_year_end,time_hour_sin,time_hour_cos
6361,โรงเรียนท่าข้ามวิทยา,860206,ท่าข้าม,8602,ท่าแซะ,86,ชุมพร,10.579849,99.113146,2022-05-02 00:00:00+07:00,...,0,2,0,0,0,0,0,0,0.000000,1.000000
8429,โรงเรียนบ้านนา,550404,สถาน,5504,นาน้อย,55,น่าน,18.241106,100.690577,2022-05-02 00:00:00+07:00,...,0,2,0,0,0,0,0,0,0.000000,1.000000
1678,บ้านนา_2,300903,กำปัง,3009,โนนไทย,30,นครราชสีมา,15.112831,102.052114,2022-05-02 00:00:00+07:00,...,0,2,0,0,0,0,0,0,0.000000,1.000000
11871,โรงเรียนสามเสนนอก(ประชาราษฎร์อนุกูล),102601,ดินแดง,1026,ดินแดง,10,กรุงเทพมหานคร,13.777972,100.569662,2022-05-02 00:00:00+07:00,...,0,2,0,0,0,0,0,0,0.000000,1.000000
13614,โรงเรียนหนองสูงสามัคคีวิทยา_2,490706,หนองสูงเหนือ,4907,หนองสูง,49,มุกดาหาร,16.494229,104.350891,2022-05-02 00:00:00+07:00,...,0,2,0,0,0,0,0,0,0.000000,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55,บ้านนา_2,300903,กำปัง,3009,โนนไทย,30,นครราชสีมา,15.112831,102.052114,2022-07-31 21:00:00+07:00,...,1,3,0,1,0,0,0,0,-0.707107,0.707107
10275,โรงเรียนสามเสนนอก(ประชาราษฎร์อนุกูล),102601,ดินแดง,1026,ดินแดง,10,กรุงเทพมหานคร,13.777972,100.569662,2022-07-31 21:00:00+07:00,...,1,3,0,1,0,0,0,0,-0.707107,0.707107
1806,บ้านนาสะแบง_2,380704,นาสะแบง,3807,ศรีวิไล,38,บึงกาฬ,18.142499,103.806521,2022-07-31 21:00:00+07:00,...,1,3,0,1,0,0,0,0,-0.707107,0.707107
8545,โรงเรียนสรรพวิทยาคม,630601,แม่สอด,6306,แม่สอด,63,ตาก,16.712990,98.573417,2022-07-31 21:00:00+07:00,...,1,3,0,1,0,0,0,0,-0.707107,0.707107


In [130]:
df_train = df_train.drop(columns=[
    "time_year", "time_is_year_start", "time_is_year_end"
])

df_train

,station_name,tambon_code,tambon_namt,amphur_code,amphur_namt,province_code,province_namt,latitude,longitude,time,...,time_weekofyear,time_quarter,time_is_weekend,time_season,time_is_month_start,time_is_month_end,time_is_quarter_start,time_is_quarter_end,time_hour_sin,time_hour_cos
6361,โรงเรียนท่าข้ามวิทยา,860206,ท่าข้าม,8602,ท่าแซะ,86,ชุมพร,10.579849,99.113146,2022-05-02 00:00:00+07:00,...,18,2,0,2,0,0,0,0,0.000000,1.000000
8429,โรงเรียนบ้านนา,550404,สถาน,5504,นาน้อย,55,น่าน,18.241106,100.690577,2022-05-02 00:00:00+07:00,...,18,2,0,2,0,0,0,0,0.000000,1.000000
1678,บ้านนา_2,300903,กำปัง,3009,โนนไทย,30,นครราชสีมา,15.112831,102.052114,2022-05-02 00:00:00+07:00,...,18,2,0,2,0,0,0,0,0.000000,1.000000
11871,โรงเรียนสามเสนนอก(ประชาราษฎร์อนุกูล),102601,ดินแดง,1026,ดินแดง,10,กรุงเทพมหานคร,13.777972,100.569662,2022-05-02 00:00:00+07:00,...,18,2,0,2,0,0,0,0,0.000000,1.000000
13614,โรงเรียนหนองสูงสามัคคีวิทยา_2,490706,หนองสูงเหนือ,4907,หนองสูง,49,มุกดาหาร,16.494229,104.350891,2022-05-02 00:00:00+07:00,...,18,2,0,2,0,0,0,0,0.000000,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55,บ้านนา_2,300903,กำปัง,3009,โนนไทย,30,นครราชสีมา,15.112831,102.052114,2022-07-31 21:00:00+07:00,...,30,3,1,3,0,1,0,0,-0.707107,0.707107
10275,โรงเรียนสามเสนนอก(ประชาราษฎร์อนุกูล),102601,ดินแดง,1026,ดินแดง,10,กรุงเทพมหานคร,13.777972,100.569662,2022-07-31 21:00:00+07:00,...,30,3,1,3,0,1,0,0,-0.707107,0.707107
1806,บ้านนาสะแบง_2,380704,นาสะแบง,3807,ศรีวิไล,38,บึงกาฬ,18.142499,103.806521,2022-07-31 21:00:00+07:00,...,30,3,1,3,0,1,0,0,-0.707107,0.707107
8545,โรงเรียนสรรพวิทยาคม,630601,แม่สอด,6306,แม่สอด,63,ตาก,16.712990,98.573417,2022-07-31 21:00:00+07:00,...,30,3,1,3,0,1,0,0,-0.707107,0.707107


In [131]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13635 entries, 6361 to 6830
Data columns (total 35 columns):
 #   Column                 Non-Null Count  Dtype                    
---  ------                 --------------  -----                    
 0   station_name           13635 non-null  object                   
 1   tambon_code            13635 non-null  int64                    
 2   tambon_namt            13635 non-null  object                   
 3   amphur_code            13635 non-null  int64                    
 4   amphur_namt            13635 non-null  object                   
 5   province_code          13635 non-null  int64                    
 6   province_namt          13635 non-null  object                   
 7   latitude               13635 non-null  float64                  
 8   longitude              13635 non-null  float64                  
 9   time                   13635 non-null  datetime64[ns, UTC+07:00]
 10  humid                  13635 non-null  float64   

In [132]:
df_train.columns

Index(['station_name', 'tambon_code', 'tambon_namt', 'amphur_code',
       'amphur_namt', 'province_code', 'province_namt', 'latitude',
       'longitude', 'time', 'humid', 'light', 'pm10', 'pm2.5', 'rainfall',
       'wind_direct', 'wind_speed', 'temp', 'time_month', 'time_day',
       'time_hour', 'time_minute', 'time_second', 'time_dayofweek',
       'time_dayofyear', 'time_weekofyear', 'time_quarter', 'time_is_weekend',
       'time_season', 'time_is_month_start', 'time_is_month_end',
       'time_is_quarter_start', 'time_is_quarter_end', 'time_hour_sin',
       'time_hour_cos'],
      dtype='object')

In [133]:
df_train = df_train.reset_index()

## Time Series Features

In [134]:
def time_series_features_rolling(df: pd.DataFrame,
                                 list_features_target_cols: list[str],
                                 list_window_size : list[int],
                                 list_aggregrate_function : list[str]):
  """
  Time Series Features Rolling :
  input ->
  - df
  - features_target_cols : list of target columns
  - window_size : window size for rolling calculations (e.g., [7, 30] for 7-day and 30-day windows).
  - aggregate_function : list of aggregate functions includes ['mean', 'std', 'min', 'max', 'median']
  return df with lagged features (nan will not be removed, please used df.dropna() after)
  """
  df_copy = df.copy()

  for feature_col in list_features_target_cols:
    for window in list_window_size:
      for agg_function in list_aggregrate_function:
        df_copy[f'rolling_{feature_col}_{window}_{agg_function}'] = df_copy[feature_col].rolling(window=window).agg(agg_function)

  return df_copy

In [135]:
def time_series_features_differencing(df: pd.DataFrame,
                                      list_features_target_cols: list[str],
                                      list_interval: list[int]):
  """
  Time Series Features Differencing:
  input ->
  - df (pd.DataFrame): The input DataFrame. It should have a DatetimeIndex for proper time-series operations,
                        though pandas' .diff() works on numerical indices too.
  - list_features_target_cols (List[str]): List of column names on which to apply differencing.
  - list_interval (List[int]): List of intervals (periods) for differencing (e.g., [1, 7, 12]).

  return ->
  pd.DataFrame: A new DataFrame with the added differenced features.
                NaN values will be present at the beginning of the series due to differencing.
                Please use df.dropna() after this operation if needed.
  """
  df_copy = df.copy()

  for feature_col in list_features_target_cols:
      for interval in list_interval:
          df_copy[f'{feature_col}_diff_{interval}'] = df_copy[feature_col].diff(periods=interval)

  return df_copy

In [136]:
def timeseries_features_lag(df: pd.DataFrame,
                            list_features_target_cols: list[str],
                            list_lags: list[int]):
  """
  Time Series Features Lagging :
  input -> df and list of target columns
  return df with lagged features (nan will not be removed, please used df.dropna() after)
  """
  df_copy = df.copy()

  for feature_col in list_features_target_cols:
    for lag in list_lags:
      df_copy[f'lag_{feature_col}_{lag}'] = df_copy[feature_col].shift(lag)

  return df_copy

In [137]:
df_train["tambon_code"].unique()

array([860206, 550404, 300903, 102601, 490706, 380704, 330802, 630601])

In [138]:
df_train

,index,station_name,tambon_code,tambon_namt,amphur_code,amphur_namt,province_code,province_namt,latitude,longitude,...,time_weekofyear,time_quarter,time_is_weekend,time_season,time_is_month_start,time_is_month_end,time_is_quarter_start,time_is_quarter_end,time_hour_sin,time_hour_cos
0,6361,โรงเรียนท่าข้ามวิทยา,860206,ท่าข้าม,8602,ท่าแซะ,86,ชุมพร,10.579849,99.113146,...,18,2,0,2,0,0,0,0,0.000000,1.000000
1,8429,โรงเรียนบ้านนา,550404,สถาน,5504,นาน้อย,55,น่าน,18.241106,100.690577,...,18,2,0,2,0,0,0,0,0.000000,1.000000
2,1678,บ้านนา_2,300903,กำปัง,3009,โนนไทย,30,นครราชสีมา,15.112831,102.052114,...,18,2,0,2,0,0,0,0,0.000000,1.000000
3,11871,โรงเรียนสามเสนนอก(ประชาราษฎร์อนุกูล),102601,ดินแดง,1026,ดินแดง,10,กรุงเทพมหานคร,13.777972,100.569662,...,18,2,0,2,0,0,0,0,0.000000,1.000000
4,13614,โรงเรียนหนองสูงสามัคคีวิทยา_2,490706,หนองสูงเหนือ,4907,หนองสูง,49,มุกดาหาร,16.494229,104.350891,...,18,2,0,2,0,0,0,0,0.000000,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13630,55,บ้านนา_2,300903,กำปัง,3009,โนนไทย,30,นครราชสีมา,15.112831,102.052114,...,30,3,1,3,0,1,0,0,-0.707107,0.707107
13631,10275,โรงเรียนสามเสนนอก(ประชาราษฎร์อนุกูล),102601,ดินแดง,1026,ดินแดง,10,กรุงเทพมหานคร,13.777972,100.569662,...,30,3,1,3,0,1,0,0,-0.707107,0.707107
13632,1806,บ้านนาสะแบง_2,380704,นาสะแบง,3807,ศรีวิไล,38,บึงกาฬ,18.142499,103.806521,...,30,3,1,3,0,1,0,0,-0.707107,0.707107
13633,8545,โรงเรียนสรรพวิทยาคม,630601,แม่สอด,6306,แม่สอด,63,ตาก,16.712990,98.573417,...,30,3,1,3,0,1,0,0,-0.707107,0.707107


In [139]:
list_tambon = [860206, 550404, 300903, 102601, 490706, 380704, 330802, 630601]

for tambon in list_tambon:

  print(df_train[ df_train["tambon_code"] == tambon ])

  # Get the unique tambon
  df = df_train[ df_train["tambon_code"] == tambon ]
  df = df.sort_values(by="time")

  # Apply Time Series Function
  df = time_series_features_rolling(
      df=df,
      list_features_target_cols=["pm10", "pm2.5", "rainfall", "wind_speed", "light"],
      list_window_size=[1, 3, 7, 15, 30],
      list_aggregrate_function=['mean', 'std', 'min', 'max', 'median']
  )

  df = timeseries_features_lag(
      df=df,
      list_features_target_cols=["pm10", "pm2.5", "rainfall", "wind_speed", "light"],
      list_lags=[1, 3, 7, 15, 30],
  )

  df = time_series_features_differencing(
      df=df,
      list_features_target_cols=["pm10", "pm2.5", "rainfall", "wind_speed", "light"],
      list_interval=[1, 3, 7, 15, 30],
  )

  # df_train = pd.merge(
  #     df_train,
  #     df,
  #     on = "index",
  #     how="left"
  # )


       index          station_name  tambon_code tambon_namt  amphur_code  \
0       6361  โรงเรียนท่าข้ามวิทยา       860206     ท่าข้าม         8602   
9       6641  โรงเรียนท่าข้ามวิทยา       860206     ท่าข้าม         8602   
16      5392  โรงเรียนท่าข้ามวิทยา       860206     ท่าข้าม         8602   
20      6680  โรงเรียนท่าข้ามวิทยา       860206     ท่าข้าม         8602   
28      5700  โรงเรียนท่าข้ามวิทยา       860206     ท่าข้าม         8602   
...      ...                   ...          ...         ...          ...   
13595   5132  โรงเรียนท่าข้ามวิทยา       860206     ท่าข้าม         8602   
13602   5844  โรงเรียนท่าข้ามวิทยา       860206     ท่าข้าม         8602   
13609   6604  โรงเรียนท่าข้ามวิทยา       860206     ท่าข้าม         8602   
13616   6683  โรงเรียนท่าข้ามวิทยา       860206     ท่าข้าม         8602   
13624   6345  โรงเรียนท่าข้ามวิทยา       860206     ท่าข้าม         8602   

      amphur_namt  province_code province_namt   latitude  longitude  ...  \
0         

/tmp/ipython-input-134-1059688766.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_copy[f'rolling_{feature_col}_{window}_{agg_function}'] = df_copy[feature_col].rolling(window=window).agg(agg_function)
/tmp/ipython-input-134-1059688766.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_copy[f'rolling_{feature_col}_{window}_{agg_function}'] = df_copy[feature_col].rolling(window=window).agg(agg_function)
/tmp/ipython-input-134-1059688766.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the re


       index                          station_name  tambon_code tambon_namt  \
3      11871  โรงเรียนสามเสนนอก(ประชาราษฎร์อนุกูล)       102601      ดินแดง   
14     10553  โรงเรียนสามเสนนอก(ประชาราษฎร์อนุกูล)       102601      ดินแดง   
19     10659  โรงเรียนสามเสนนอก(ประชาราษฎร์อนุกูล)       102601      ดินแดง   
24     11887  โรงเรียนสามเสนนอก(ประชาราษฎร์อนุกูล)       102601      ดินแดง   
40     11120  โรงเรียนสามเสนนอก(ประชาราษฎร์อนุกูล)       102601      ดินแดง   
...      ...                                   ...          ...         ...   
13601  11017  โรงเรียนสามเสนนอก(ประชาราษฎร์อนุกูล)       102601      ดินแดง   
13605  11824  โรงเรียนสามเสนนอก(ประชาราษฎร์อนุกูล)       102601      ดินแดง   
13619  10828  โรงเรียนสามเสนนอก(ประชาราษฎร์อนุกูล)       102601      ดินแดง   
13627  11585  โรงเรียนสามเสนนอก(ประชาราษฎร์อนุกูล)       102601      ดินแดง   
13631  10275  โรงเรียนสามเสนนอก(ประชาราษฎร์อนุกูล)       102601      ดินแดง   

       amphur_code amphur_namt  province_code  pro

/tmp/ipython-input-134-1059688766.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_copy[f'rolling_{feature_col}_{window}_{agg_function}'] = df_copy[feature_col].rolling(window=window).agg(agg_function)
/tmp/ipython-input-134-1059688766.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_copy[f'rolling_{feature_col}_{window}_{agg_function}'] = df_copy[feature_col].rolling(window=window).agg(agg_function)
/tmp/ipython-input-134-1059688766.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the re

       index   station_name  tambon_code tambon_namt  amphur_code amphur_namt  \
5       3402  บ้านนาสะแบง_2       380704     นาสะแบง         3807     ศรีวิไล   
13      3007  บ้านนาสะแบง_2       380704     นาสะแบง         3807     ศรีวิไล   
17      2117  บ้านนาสะแบง_2       380704     นาสะแบง         3807     ศรีวิไล   
23      1729  บ้านนาสะแบง_2       380704     นาสะแบง         3807     ศรีวิไล   
33      1961  บ้านนาสะแบง_2       380704     นาสะแบง         3807     ศรีวิไล   
...      ...            ...          ...         ...          ...         ...   
13603   2548  บ้านนาสะแบง_2       380704     นาสะแบง         3807     ศรีวิไล   
13610   3355  บ้านนาสะแบง_2       380704     นาสะแบง         3807     ศรีวิไล   
13613   2359  บ้านนาสะแบง_2       380704     นาสะแบง         3807     ศรีวิไล   
13621   3114  บ้านนาสะแบง_2       380704     นาสะแบง         3807     ศรีวิไล   
13632   1806  บ้านนาสะแบง_2       380704     นาสะแบง         3807     ศรีวิไล   

       province_code provin

/tmp/ipython-input-134-1059688766.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_copy[f'rolling_{feature_col}_{window}_{agg_function}'] = df_copy[feature_col].rolling(window=window).agg(agg_function)
/tmp/ipython-input-134-1059688766.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_copy[f'rolling_{feature_col}_{window}_{agg_function}'] = df_copy[feature_col].rolling(window=window).agg(agg_function)
/tmp/ipython-input-134-1059688766.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the re

       index         station_name  tambon_code tambon_namt  amphur_code  \
7      10145  โรงเรียนสรรพวิทยาคม       630601      แม่สอด         6306   
15      9248  โรงเรียนสรรพวิทยาคม       630601      แม่สอด         6306   
25      8484  โรงเรียนสรรพวิทยาคม       630601      แม่สอด         6306   
38      8620  โรงเรียนสรรพวิทยาคม       630601      แม่สอด         6306   
47     10002  โรงเรียนสรรพวิทยาคม       630601      แม่สอด         6306   
...      ...                  ...          ...         ...          ...   
13598   9283  โรงเรียนสรรพวิทยาคม       630601      แม่สอด         6306   
13611  10099  โรงเรียนสรรพวิทยาคม       630601      แม่สอด         6306   
13612   9100  โรงเรียนสรรพวิทยาคม       630601      แม่สอด         6306   
13622   9845  โรงเรียนสรรพวิทยาคม       630601      แม่สอด         6306   
13633   8545  โรงเรียนสรรพวิทยาคม       630601      แม่สอด         6306   

      amphur_namt  province_code province_namt  latitude  longitude  ...  \
7          แม่สอด      

/tmp/ipython-input-134-1059688766.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_copy[f'rolling_{feature_col}_{window}_{agg_function}'] = df_copy[feature_col].rolling(window=window).agg(agg_function)
/tmp/ipython-input-134-1059688766.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_copy[f'rolling_{feature_col}_{window}_{agg_function}'] = df_copy[feature_col].rolling(window=window).agg(agg_function)
/tmp/ipython-input-134-1059688766.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the re

### Visualize Dataset

In [ ]:
df_eda = df_train.copy()

In [ ]:
df_eda

In [ ]:
def plot_violin_box_distribution(df, columns):
    """
    Plots violin plots, box plots, and distribution plots for specified columns.

    Parameters:
        df (pd.DataFrame): The input DataFrame.
        columns (list): List of columns to plot.
    """
    # Filter columns that exist in the DataFrame
    valid_columns = [col for col in columns if col in df.columns]

    # Calculate the number of rows needed (3 plots per column)
    num_columns = len(valid_columns)
    num_rows = num_columns  # One row per column
    fig, axes = plt.subplots(num_rows, 3, figsize=(18, 5 * num_rows))

    # If only one column, axes will be 1D, so reshape it
    if num_columns == 1:
        axes = axes.reshape(1, -1)

    # Loop through the valid columns and plot
    for i, col in enumerate(valid_columns):
        # Violin Plot
        sns.violinplot(data=df, x=col, ax=axes[i, 0], palette='viridis')
        axes[i, 0].set_title(f'Violin Plot of {col}')
        axes[i, 0].set_xlabel(col)

        # Box Plot
        sns.boxplot(data=df, x=col, ax=axes[i, 1], palette='viridis')
        axes[i, 1].set_title(f'Box Plot of {col}')
        axes[i, 1].set_xlabel(col)

        # Distribution Plot (Histogram with KDE)
        sns.histplot(data=df, x=col, kde=True, ax=axes[i, 2], color='skyblue')
        axes[i, 2].set_title(f'Distribution of {col}')
        axes[i, 2].set_xlabel(col)
        axes[i, 2].set_ylabel('Density')

    # Adjust layout and display
    plt.tight_layout()
    plt.show()

In [ ]:
plot_violin_box_distribution(df_eda, columns=["temp", "pm2.5", "wind_speed"])

In [ ]:
plot_violin_box_distribution(df_eda, columns=["pm10", "rainfall", "wind_direct"])

In [ ]:
df_eda["pm2.5"].describe()

In [ ]:
df_eda["pm10"].describe()

In [ ]:
print(len(df_eda[df_eda['pm2.5'] >= 50]))
print(len(df_eda[df_eda['pm10'] >= 50]))

In [ ]:
df_eda['pm2.5'] = df_eda['pm2.5'].clip(6, 50)
df_eda['pm10'] = df_eda['pm10'].clip(6, 50)

In [ ]:
df_eda.info()

In [ ]:
df_eda = df_eda.drop(
    columns = [
        "tambon_code", "amphur_code", "province_code",
    ]
)

df_eda.describe()

### Check Duplicate Columns

In [ ]:
from collections import defaultdict

renamer = defaultdict()
renamer

In [ ]:
for column_name in df_eda.columns[df_eda.columns.duplicated(keep=False)].tolist():
    if column_name not in renamer:
      print(f"columns : {column_name}")
      #renamer[column_name] = [column_name+'_0']

    else:
      continue
      # renamer[column_name].append(column_name +'_'+str(len(renamer[column_name])))

In [ ]:
nan_rows = df_eda[df_eda.isnull().T.any()]
nan_rows

### Interpolation Fill Nan

In [ ]:
# Fill NaN values in 'PM2.5' column using interpolation in the new DataFrame
df_eda['pm2.5'] = df_eda['pm2.5'].interpolate(method='linear')
df_eda['pm10'] = df_eda['pm10'].interpolate(method='linear')
df_eda['temp'] = df_eda['temp'].interpolate(method='linear')

In [ ]:
nan_rows = df_eda[df_eda.isnull().T.any()]
nan_rows

In [ ]:
df_eda = df_eda.dropna()

nan_rows = df_eda[df_eda.isnull().T.any()]
nan_rows

## AutoGluon Tabular Predict

In [ ]:
!pip install autogluon

In [ ]:
from autogluon.tabular import TabularDataset, TabularPredictor
from autogluon.common import space

train_data = df_eda.copy()
label = 'temp'

# From hackathon
metric = 'mae'

In [ ]:
# nn_options = {  # specifies non-default hyperparameter values for neural network models
#     'num_epochs': 100,  # number of training epochs (controls training time of NN models)
#     'learning_rate': space.Real(1e-4, 1e-2, default=5e-4, log=True),  # learning rate used in training (real-valued hyperparameter searched on log-scale)
#     'activation': space.Categorical('relu', 'softrelu', 'tanh'),  # activation function used in NN (categorical hyperparameter, default = first entry)
#     'dropout_prob': space.Real(0.0, 0.5, default=0.1),  # dropout probability (real-valued hyperparameter)
# }

# gbm_options = {  # specifies non-default hyperparameter values for lightGBM gradient boosted trees
#     'num_boost_round': 100,  # number of boosting rounds (controls training time of GBM models)
#     'num_leaves': space.Int(lower=26, upper=66, default=36),  # number of leaves in trees (integer hyperparameter)
# }

#  # NOTE: comment this line out if you get errors on Mac OSX
# hyperparameters = { # hyperparameters of each model type
#                    'GBM': gbm_options,
#                     # NOTE: comment this line out if you get errors on Mac OSX
#                    'NN_TORCH': nn_options,
#                   }

# time_limit = 10*60  # train various models for ~2 min
# num_trials = 5  # try at most 5 different hyperparameter configurations for each type of model
# search_strategy = 'auto'  # to tune hyperparameters using random search routine with a local scheduler

# hyperparameter_tune_kwargs = {  # HPO is not performed unless hyperparameter_tune_kwargs is specified
#     'num_trials': num_trials,
#     'scheduler' : 'local',
#     'searcher': search_strategy,
# }

predictor = TabularPredictor(label=label, eval_metric=metric).fit(
    train_data,
    # time_limit=time_limit,
    # hyperparameters=hyperparameters,
    # hyperparameter_tune_kwargs=hyperparameter_tune_kwargs,
    presets='best_quality',
)

No path specified. Models will be saved in: "AutogluonModels/ag-20250705_092231"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.11.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Sun Mar 30 16:01:29 UTC 2025
CPU Count:          12
Memory Avail:       50.77 GB / 52.96 GB (95.9%)
Disk Space Avail:   192.89 GB / 235.68 GB (81.8%)
Presets specified: ['best_quality']
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the optimal `num_stack_levels` value. Copies of AutoGluon will 

(_dystack pid=9043) ╭──────────────────────────────────────────────────────────╮
(_dystack pid=9043) │ Configuration for experiment     NeuralNetTorch_BAG_L1   │
(_dystack pid=9043) ├──────────────────────────────────────────────────────────┤
(_dystack pid=9043) │ Search algorithm                 SearchGenerator         │
(_dystack pid=9043) │ Scheduler                        FIFOScheduler           │
(_dystack pid=9043) │ Number of trials                 5                       │
(_dystack pid=9043) ╰──────────────────────────────────────────────────────────╯
(_dystack pid=9043) 
(_dystack pid=9043) View detailed results here: /content/AutogluonModels/ag-20250705_092231/ds_sub_fit/sub_fit_ho/models/NeuralNetTorch_BAG_L1


(_ray_fit pid=11890) 	Ran out of time, stopping training early. (Stopping on epoch 96)
(_dystack pid=9043) Reached timeout of 44.07339190067053 seconds. Stopping all trials.
(_dystack pid=9043) Wrote the latest version of all result files and experiment state to '/content/AutogluonModels/ag-20250705_092231/ds_sub_fit/sub_fit_ho/models/NeuralNetTorch_BAG_L1' in 0.0069s.
(_ray_fit pid=11888) 	Ran out of time, stopping training early. (Stopping on epoch 99) [repeated 2x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)
(_dystack pid=9043) Failed to fetch metrics for 3 trial(s):
(_dystack pid=9043) - 1e89b1cf: FileNotFoundError('Could not fetch metrics for 1e89b1cf: both result.json and progress.csv were not found at /content/AutogluonModels/ag-20250705_092231/ds_sub_fit/sub_fit_ho/models/NeuralNetTorch_BAG_L1/1e8

(_dystack pid=9043) 


 20%|██        | 1/5 [00:05<00:20,  5.15s/it]
(_dystack pid=9043) 	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (8 workers, per: cpus=1, gpus=0, memory=0.03%)
 40%|████      | 2/5 [00:10<00:15,  5.13s/it]
(_dystack pid=9043) 	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (8 workers, per: cpus=1, gpus=0, memory=0.05%)
100%|██████████| 5/5 [00:26<00:00,  5.24s/it]
(_dystack pid=9043) Fitted model: LightGBM_BAG_L2/T1 ...
(_dystack pid=9043) 	-0.5392	 = Validation score   (-mean_absolute_error)
(_dystack pid=9043) 	5.15s	 = Training   runtime
(_dystack pid=9043) 	0.12s	 = Validation runtime
(_dystack pid=9043) Fitted model: LightGBM_BAG_L2/T2 ...
(_dystack pid=9043) 	-0.5306	 = Validation score   (-mean_absolute_error)
(_dystack pid=9043) 	5.1s	 = Training   runtime
(_dystack pid=9043) 	0.1s	 = Validation runtime
(_dystack pid=9043) Fitted model: LightGBM_BAG_L2/T3 ...
(_dystack pid=9043) 	-0.538	 = Validation s

...

2025-07-05 09:27:40,668	INFO timeout.py:54 -- Reached timeout of 136.35051085336804 seconds. Stopping all trials.
2025-07-05 09:27:40,677	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to '/content/AutogluonModels/ag-20250705_092231/models/NeuralNetTorch_BAG_L1' in 0.0072s.
2025-07-05 09:27:42,093	WARNING experiment_analysis.py:180 -- Failed to fetch metrics for 4 trial(s):
- 305d92ce: FileNotFoundError('Could not fetch metrics for 305d92ce: both result.json and progress.csv were not found at /content/AutogluonModels/ag-20250705_092231/models/NeuralNetTorch_BAG_L1/305d92ce')
- 0aca1bca: FileNotFoundError('Could not fetch metrics for 0aca1bca: both result.json and progress.csv were not found at /content/AutogluonModels/ag-20250705_092231/models/NeuralNetTorch_BAG_L1/0aca1bca')
- 97215195: FileNotFoundError('Could not fetch metrics for 97215195: both result.json and progress.csv were not found at /content/AutogluonModels/ag-20250705_092231/models/N

  0%|          | 0/5 [00:00<?, ?it/s]

	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (8 workers, per: cpus=1, gpus=0, memory=0.04%)
	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (8 workers, per: cpus=1, gpus=0, memory=0.04%)
	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (8 workers, per: cpus=1, gpus=0, memory=0.05%)
	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (8 workers, per: cpus=1, gpus=0, memory=0.04%)
	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (8 workers, per: cpus=1, gpus=0, memory=0.04%)
Fitted model: LightGBM_BAG_L2/T1 ...
	-0.4994	 = Validation score   (-mean_absolute_error)
	5.36s	 = Training   runtime
	0.13s	 = Validation runtime
Fitted model: LightGBM_BAG_L2/T2 ...
	-0.4943	 = Validation score   (-mean_absolute_error)
	5.15s	 = Training   runtime
	0.1s	 = Validation runtime
Fitted model: LightGBM_BAG_L2/T3 ...
	

+----------------------------------------------------------+
| Configuration for experiment     NeuralNetTorch_BAG_L2   |
+----------------------------------------------------------+
| Search algorithm                 SearchGenerator         |
| Scheduler                        FIFOScheduler           |
| Number of trials                 5                       |
+----------------------------------------------------------+

View detailed results here: /content/AutogluonModels/ag-20250705_092231/models/NeuralNetTorch_BAG_L2


2025-07-05 09:30:19,547	INFO timeout.py:54 -- Reached timeout of 130.37792751789092 seconds. Stopping all trials.
2025-07-05 09:30:19,556	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to '/content/AutogluonModels/ag-20250705_092231/models/NeuralNetTorch_BAG_L2' in 0.0081s.
2025-07-05 09:30:20,128	WARNING experiment_analysis.py:180 -- Failed to fetch metrics for 5 trial(s):
- 159261aa: FileNotFoundError('Could not fetch metrics for 159261aa: both result.json and progress.csv were not found at /content/AutogluonModels/ag-20250705_092231/models/NeuralNetTorch_BAG_L2/159261aa')
- 07553bf1: FileNotFoundError('Could not fetch metrics for 07553bf1: both result.json and progress.csv were not found at /content/AutogluonModels/ag-20250705_092231/models/NeuralNetTorch_BAG_L2/07553bf1')
- a9e97997: FileNotFoundError('Could not fetch metrics for a9e97997: both result.json and progress.csv were not found at /content/AutogluonModels/ag-20250705_092231/models/N

## Transform Data Test

In [ ]:
df_test = pd.read_csv("/content/IOT_Test.csv")
df_submission = pd.read_csv("/content/IOT_Submit.csv")
df_test

,id,mac,station_name,tambon_code,tambon_namt,amphur_code,amphur_namt,province_code,province_namt,latitude,longitude,time,humid,light,pm10,pm2.5,rainfall,wind_direct,wind_speed
0,1,3C71BF18EA64,บ้านนา_2,300903,กำปัง,3009,โนนไทย,30,นครราชสีมา,15.112831,102.052114,2022-05-23 04:00:00+07:00,88.9,1.0,NaN,NaN,0.0,180.0,2.3
1,2,3C71BF18EA64,บ้านนา_2,300903,กำปัง,3009,โนนไทย,30,นครราชสีมา,15.112831,102.052114,2022-05-12 16:00:00+07:00,85.0,40.0,NaN,NaN,0.3,135.0,1.5
2,3,3C71BF18EA64,บ้านนา_2,300903,กำปัง,3009,โนนไทย,30,นครราชสีมา,15.112831,102.052114,2022-06-20 03:00:00+07:00,88.9,7.0,NaN,NaN,0.0,157.5,0.0
3,4,3C71BF18EA64,บ้านนา_2,300903,กำปัง,3009,โนนไทย,30,นครราชสีมา,15.112831,102.052114,2022-07-13 13:00:00+07:00,81.9,66.0,NaN,NaN,0.0,135.0,0.2
4,5,3C71BF18EA64,บ้านนา_2,300903,กำปัง,3009,โนนไทย,30,นครราชสีมา,15.112831,102.052114,2022-07-05 09:00:00+07:00,64.9,71.0,NaN,NaN,0.0,225.0,2.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3408,3409,3C71BF18CEA4,โรงเรียนหนองสูงสามัคคีวิทยา_2,490706,หนองสูงเหนือ,4907,หนองสูง,49,มุกดาหาร,16.494229,104.350891,2022-06-05 03:00:00+07:00,87.0,0.0,NaN,NaN,0.0,315.0,0.0
3409,3410,3C71BF18CEA4,โรงเรียนหนองสูงสามัคคีวิทยา_2,490706,หนองสูงเหนือ,4907,หนองสูง,49,มุกดาหาร,16.494229,104.350891,2022-05-26 09:00:00+07:00,71.7,81.0,NaN,NaN,0.0,180.0,3.1
3410,3411,3C71BF18CEA4,โรงเรียนหนองสูงสามัคคีวิทยา_2,490706,หนองสูงเหนือ,4907,หนองสูง,49,มุกดาหาร,16.494229,104.350891,2022-05-21 22:00:00+07:00,86.5,2.0,NaN,NaN,0.0,0.0,0.6
3411,3412,3C71BF18CEA4,โรงเรียนหนองสูงสามัคคีวิทยา_2,490706,หนองสูงเหนือ,4907,หนองสูง,49,มุกดาหาร,16.494229,104.350891,2022-07-21 01:00:00+07:00,96.5,0.0,NaN,NaN,0.0,0.0,0.0


In [ ]:
df_test["temp"] = np.nan

In [ ]:
# Format Time
df_test["time"] = pd.to_datetime(df_test["time"])
df_test = df_test.sort_values(by="id")
df_test = extract_time_features(df=df_test, time_columns=["time"])
df_test = df_test.drop(columns=["time", "time_dt", "mac", "time_year", "time_is_year_start", "time_is_year_end"])
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3413 entries, 0 to 3412
Data columns (total 35 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     3413 non-null   int64  
 1   station_name           3413 non-null   object 
 2   tambon_code            3413 non-null   int64  
 3   tambon_namt            3413 non-null   object 
 4   amphur_code            3413 non-null   int64  
 5   amphur_namt            3413 non-null   object 
 6   province_code          3413 non-null   int64  
 7   province_namt          3413 non-null   object 
 8   latitude               3413 non-null   float64
 9   longitude              3413 non-null   float64
 10  humid                  3413 non-null   float64
 11  light                  3413 non-null   float64
 12  pm10                   1687 non-null   float64
 13  pm2.5                  1687 non-null   float64
 14  rainfall               3413 non-null   float64
 15  wind

/tmp/ipython-input-9-1301132536.py:61: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df_copy[f'{col}_dt'] = pd.to_datetime(df_copy[col], errors='coerce', infer_datetime_format=True)


In [ ]:
# Clip
df_test['pm2.5'] = df_test['pm2.5'].clip(6, 50)
df_test['pm10'] = df_test['pm10'].clip(6, 50)

# Interpolation
df_test['pm2.5'] = df_test['pm2.5'].interpolate(method='linear')
df_test['pm10'] = df_test['pm10'].interpolate(method='linear')

In [ ]:
df_test

,id,station_name,tambon_code,tambon_namt,amphur_code,amphur_namt,province_code,province_namt,latitude,longitude,...,time_weekofyear,time_quarter,time_is_weekend,time_season,time_is_month_start,time_is_month_end,time_is_quarter_start,time_is_quarter_end,time_hour_sin,time_hour_cos
0,1,บ้านนา_2,300903,กำปัง,3009,โนนไทย,30,นครราชสีมา,15.112831,102.052114,...,21,2,0,2,0,0,0,0,0.866025,0.500000
1,2,บ้านนา_2,300903,กำปัง,3009,โนนไทย,30,นครราชสีมา,15.112831,102.052114,...,19,2,0,2,0,0,0,0,-0.866025,-0.500000
2,3,บ้านนา_2,300903,กำปัง,3009,โนนไทย,30,นครราชสีมา,15.112831,102.052114,...,25,2,0,3,0,0,0,0,0.707107,0.707107
3,4,บ้านนา_2,300903,กำปัง,3009,โนนไทย,30,นครราชสีมา,15.112831,102.052114,...,28,3,0,3,0,0,0,0,-0.258819,-0.965926
4,5,บ้านนา_2,300903,กำปัง,3009,โนนไทย,30,นครราชสีมา,15.112831,102.052114,...,27,3,0,3,0,0,0,0,0.707107,-0.707107
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3408,3409,โรงเรียนหนองสูงสามัคคีวิทยา_2,490706,หนองสูงเหนือ,4907,หนองสูง,49,มุกดาหาร,16.494229,104.350891,...,22,2,1,3,0,0,0,0,0.707107,0.707107
3409,3410,โรงเรียนหนองสูงสามัคคีวิทยา_2,490706,หนองสูงเหนือ,4907,หนองสูง,49,มุกดาหาร,16.494229,104.350891,...,21,2,0,2,0,0,0,0,0.707107,-0.707107
3410,3411,โรงเรียนหนองสูงสามัคคีวิทยา_2,490706,หนองสูงเหนือ,4907,หนองสูง,49,มุกดาหาร,16.494229,104.350891,...,20,2,1,2,0,0,0,0,-0.500000,0.866025
3411,3412,โรงเรียนหนองสูงสามัคคีวิทยา_2,490706,หนองสูงเหนือ,4907,หนองสูง,49,มุกดาหาร,16.494229,104.350891,...,29,3,0,3,0,0,0,0,0.258819,0.965926


## Prediction

In [ ]:
y_pred = predictor.predict(df_test.drop(columns=[label]))
y_pred.head()

,temp
0,25.475311
1,28.583195
2,25.046091
3,29.336971
4,34.380157


In [ ]:
y_pred = y_pred.reset_index()
y_pred = y_pred.rename(columns={"index":"id"})
y_pred["id"] = y_pred["id"] + 1

In [ ]:
y_pred

,id,temp
0,1,25.475311
1,2,28.583195
2,3,25.046091
3,4,29.336971
4,5,34.380157
...,...,...
3408,3409,25.764868
3409,3410,32.715298
3410,3411,26.925190
3411,3412,24.783997


In [ ]:
df_submission

,id,temp
0,1,25.6
1,2,30.1
2,3,24.7
3,4,NaN
4,5,NaN
...,...,...
3408,3409,NaN
3409,3410,NaN
3410,3411,NaN
3411,3412,NaN


In [ ]:
y_pred.to_csv("Tabular_Gluon_TimeExtractorOnly.csv")